# Assignment 3


In [148]:
# Reused from practical
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import networkx as nx
import seaborn as sns
from itertools import product
from collections import Counter, defaultdict
import warnings
warnings.filterwarnings('ignore')

plt.style.use('default')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 8)

In [149]:
class BooleanNetwork:
    def __init__(self, node_names):
        self.nodes = {name: 0 for name in node_names}
        self.rules = {}
        self.history = []
        self.graph = nx.DiGraph()  # NetworkX graph for visualization

        # Add nodes to NetworkX graph
        self.graph.add_nodes_from(node_names)

    def add_rule(self, target_node, rule_function, rule_description=""):
        """
        Add Boolean rule for a node

        Args:
            target_node: Node to update
            rule_function: Function that takes current state dict and returns True/False
            rule_description: Human-readable description
        """
        self.rules[target_node] = {
            'function': rule_function,
            'description': rule_description
        }

    def set_state(self, **kwargs):
        """Set states of specific nodes"""
        for node, value in kwargs.items():
            if node in self.nodes:
                self.nodes[node] = int(bool(value))

    def get_state_vector(self):
        """Get current state as list in sorted order"""
        return [self.nodes[node] for node in sorted(self.nodes.keys())]

    def update_synchronous(self):
        """Update all nodes simultaneously"""
        new_state = {}
        for node in self.nodes:
            if node in self.rules:
                new_state[node] = int(self.rules[node]['function'](self.nodes))
            else:
                new_state[node] = self.nodes[node]  # No rule = no change

        self.nodes = new_state
        self.history.append(self.get_state_vector())

    def simulate(self, steps=10, record_history=True):
        """Run simulation"""
        if record_history:
            self.history = [self.get_state_vector()]

        for step in range(steps):
            self.update_synchronous()

            # Check for steady state
            if len(self.history) >= 2 and self.history[-1] == self.history[-2]:
                break

        return np.array(self.history)

In [150]:
# 🟢 Create the regulatory network
nodes = ['DNA_damage', 'p53', 'MYC', 'CDK2', 'MDM2', 'p21', 'Growth', 'Death']
network = BooleanNetwork(nodes)
node_names = sorted(network.nodes.keys())
max_steps=18

def make_network():
    network = BooleanNetwork(nodes)
    # Define Boolean rules (based on real biology, simplified)
    network.add_rule('DNA_damage', lambda s: s['DNA_damage'], "DNA_damage = INPUT (constant)")
    network.add_rule('p21', lambda s: s['p53'], "p21 = p53")
    network.add_rule('MYC', lambda s: (not s['p53']) and (not s['p21']), "MYC = (NOT p53) AND (NOT p21)")
    network.add_rule('CDK2', lambda s: s['MYC'] and (not s['p21']) and (not s['p53']), "CDK2 = MYC AND (NOT p21) AND (NOT p53)")
    network.add_rule('MDM2', lambda s: s['MYC'], "MDM2 = MYC")
    network.add_rule('p53', lambda s: s['DNA_damage'] and not s['MDM2'], "p53 = DNA_damage AND (NOT MDM2)")
    network.add_rule('Growth',lambda s: s['CDK2'] and s['MYC'] and (not s['p53']), "Growth = CDK2 AND MYC AND (NOT p53)")
    network.add_rule('Death', lambda s: s['p53'] and s['DNA_damage'] and (not s['Growth']), "Death = p53 AND DNA_damage AND (NOT Growth)")
    return network

network = make_network()


In [151]:
# Reused from practical
def three_scenarios(network, steps=15):

    scenarios = {
        'Healthy Cell': {'DNA_damage': 0, 'p53': 0, 'MYC': 0, 'CDK2': 0, 'MDM2': 0, 'p21': 0, 'Growth': 0, 'Death': 0},
        'Stressed Cell': { 'DNA_damage': 1, 'p53': 0, 'MYC': 0, 'CDK2': 0, 'MDM2': 0, 'p21': 0, 'Growth': 0, 'Death': 0},
        'Oncogene Hijacked Cell': { 'DNA_damage': 0, 'p53': 0, 'MYC': 1, 'CDK2': 0, 'MDM2': 0, 'p21': 0, 'Growth': 0, 'Death': 0}
    }

    results = {}
    node_names = sorted(network.nodes.keys())  # state vectors are stored in alphabetical node order

    for scenario_name, scenario in scenarios.items():
        print(f"\n{scenario_name}")
        network.set_state(**scenario)

        trajectory = network.simulate(steps=steps)
        results[scenario_name] = trajectory  # keep full trajectory for the heatmap below

        if not np.array_equal(trajectory[-1], trajectory[-2]):
            print("  -> No fixed steady state reached (network keeps changing, possibly a limit cycle)")
            continue

        final_state = trajectory[-1]
        final_dict = {node: final_state[i] for i, node in enumerate(node_names)}

        for node, value in final_dict.items():
            print(f"  {node}: {value}")
        print(f"  -> Growth={final_dict['Growth']}, Death={final_dict['Death']}, p53={final_dict['p53']}, DNA_damage={final_dict['DNA_damage']}")
    return results

In [152]:
# Reused from practical
def visualise_final_states(results, node_names=node_names):

    n_scenarios = len(results)
    fig, axes = plt.subplots(1, n_scenarios, figsize=(5*n_scenarios, 6))

    if n_scenarios == 1:
        axes = [axes]

    for scenario_idx, (scenario_name, trajectory) in enumerate(results.items()):
        ax = axes[scenario_idx]


        trajectory_matrix = trajectory.T

        im = ax.imshow(trajectory_matrix, cmap='RdYlBu_r', aspect='auto', interpolation='nearest')

        # Formatting
        ax.set_title(f'{scenario_name}', fontweight='bold', fontsize=12)
        ax.set_xlabel('Time Steps', fontsize=10)
        ax.set_ylabel('Network Nodes', fontsize=10)
        ax.set_yticks(range(len(node_names)))
        ax.set_yticklabels(node_names, fontsize=9)

        # Add text annotations
        for t in range(trajectory.shape[0]):
            for n in range(len(node_names)):
                color = 'white' if trajectory_matrix[n, t] == 1 else 'black'
                ax.text(t, n, int(trajectory_matrix[n, t]),
                       ha="center", va="center", color=color, fontweight='bold')

        # Highlight key output nodes
        output_indices = [i for i, name in enumerate(node_names) if name in ['Growth', 'Death', 'p53']]
        for output_idx in output_indices:  # Changed variable name to avoid conflict
            ax.axhline(y=output_idx, color='red', linestyle='--', alpha=0.3, linewidth=2)

    plt.suptitle('Boolean Network Dynamics: Cell Fate Decision Making',
                 fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()

In [153]:
# 🟡 Let's find the attractor states
def find_unique_attractors(network):
    attractors = []
    max_steps=15
    node_names = sorted(network.nodes.keys())
    n_nodes = len(node_names)

    print(f"Testing all {2**n_nodes} possible initial states...")

    # TODO: Generate all possible initial states
    # Hint: Use itertools.product([0, 1], repeat=n_nodes)
    all_states = list(product([0, 1], repeat=n_nodes))

    for initial_state in all_states:
      # Set the network to this initial state
      state_dict = {node_names[i]: initial_state[i] for i in range(n_nodes)}
      network.set_state(**state_dict)

      # TODO: Simulate the network
      trajectory = network.simulate(steps=max_steps)

      # Check if it reached a steady state (last two states are the same)
      if len(trajectory) >= 2:
        final_state = tuple(int(x) for x in trajectory[-1])  # Convert to tuple of plain ints for comparison

      # Check if this is a new attractor
        if np.array_equal(trajectory[-1], trajectory[-2]):  # Steady state reached
          if final_state not in attractors:
            attractors.append(final_state)

    return attractors

attractors = find_unique_attractors(network)

Testing all 256 possible initial states...


In [154]:
# Reused from practical
def show_attractors(attractors, skipvisuals=False):
    print(f"\nFOUND {len(attractors)} ATTRACTORS:")

    for i, attractor in enumerate(attractors):
            print(f"\nAttractor {i+1}: {list(attractor)}")

            # Create state dictionary for easy access
            state_dict = {node_names[j]: attractor[j] for j in range(len(node_names))}

            # Analyze the biological meaning
            growth_active = state_dict['Growth'] == 1
            death_active = state_dict['Death'] == 1
            p53_active = state_dict['p53'] == 1
            dna_damage_active = state_dict['DNA_damage'] == 1

            print(f"   Growth: {'ON' if growth_active else 'OFF'}")
            print(f"   Death: {'ON' if death_active else 'OFF'}")
            print(f"   p53: {'ON' if p53_active else 'OFF'}")
            print(f"   DNA Damage: {'ON' if dna_damage_active else 'OFF'}")

            # TODO: Interpret the meaning
            if growth_active and not death_active and not dna_damage_active:
                interpretation = "HEALTHY PROLIFERATION - no damage, cell divides normally"
            elif death_active and not growth_active:
                interpretation = "APOPTOSIS - p53 detects DNA damage and triggers cell death"
            elif growth_active and not death_active and dna_damage_active:
                interpretation = "CANCER-LIKE - cell keeps growing despite DNA damage (MDM2 silences p53)"
            else:
                interpretation = "CONFLICT - Unusual state"

            print(f"   → {interpretation}")


    if not skipvisuals:
        # Convert attractors to clean matrix
        attractor_matrix = []
        for attractor in attractors:
            clean_row = [int(x) for x in attractor]
            attractor_matrix.append(clean_row)

        attractor_matrix = np.array(attractor_matrix)

        # Create heatmap
        plt.figure(figsize=(10, max(6, len(attractors))))

        # Create heatmap with custom colors
        ax = sns.heatmap(attractor_matrix,
                         xticklabels=node_names,
                         yticklabels=[f'Attractor {i+1}' for i in range(len(attractors))],
                         cmap='RdYlBu_r',
                         cbar_kws={'label': 'Node State (0=OFF, 1=ON)'},
                         annot=True,
                         fmt='d',
                         linewidths=0.5)

        plt.title('Boolean Network Attractors: Final States Comparison',
                  fontsize=14, fontweight='bold')
        plt.xlabel('Network Nodes', fontsize=12)
        plt.ylabel('Attractors', fontsize=12)

        # Highlight key output nodes
        output_nodes = ['Growth', 'Death', 'p53', 'DNA_damage']
        for node in output_nodes:
            if node in node_names:
                idx = node_names.index(node)
                ax.axvline(x=idx+0.5, color='red', linestyle='--', alpha=0.7, linewidth=2)

        plt.tight_layout()
        plt.show()

In [155]:
# Track which initial states lead to which attractors

def match_initial_state_to_attractor(network, attractors, skipvisuals=False):
    basin_data = defaultdict(list)  # attractor -> list of initial states
    attractor_map = {}  # initial_state -> attractor_index

    n_nodes = len(node_names)
    all_states = list(product([0, 1], repeat=n_nodes))
    print(f"Analyzing {len(all_states)} initial states...")

    for initial_state in all_states:
        # Set network state
        state_dict = {node_names[i]: initial_state[i] for i in range(n_nodes)}
        network.set_state(**state_dict)

        # Simulate
        trajectory = network.simulate(steps=max_steps, record_history=True)

        # Find which attractor this leads to
        if len(trajectory) >= 2 and np.array_equal(trajectory[-1], trajectory[-2]):
            final_state = tuple(int(x) for x in trajectory[-1])  # Clean conversion

            # Find matching attractor
            for att_idx, attractor in enumerate(attractors):
                clean_attractor = tuple(int(x) for x in attractor)
                if final_state == clean_attractor:
                    basin_data[att_idx].append(initial_state)
                    attractor_map[initial_state] = att_idx
                    break
    # Display basin sizes
    print(f"\n Basin Sizes:")
    total_states = len(all_states)
    for att_idx in range(len(attractors)):
        basin_size = len(basin_data[att_idx])
        percentage = (basin_size / total_states) * 100
        print(f"   Attractor {att_idx+1}: {basin_size:3d} states ({percentage:5.1f}%)")

    if not skipvisuals:
        # Create pie chart of basin sizes
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

        # Pie chart of basin sizes
        basin_sizes = [len(basin_data[i]) for i in range(len(attractors))]
        basin_labels = [f'Attractor {i+1}\n({size} states)' for i, size in enumerate(basin_sizes)]
        colors = plt.cm.Set3(np.linspace(0, 1, len(attractors)))

        wedges, texts, autotexts = ax1.pie(basin_sizes,
                                           labels=basin_labels,
                                           colors=colors,
                                           autopct='%1.1f%%',
                                           startangle=90)

        ax1.set_title('Basin of Attraction Sizes\n"Which attractor do most states reach?"',
                      fontsize=12, fontweight='bold')

        # Bar chart comparison
        attractor_names = [f'Attractor {i+1}' for i in range(len(attractors))]
        bars = ax2.bar(attractor_names, basin_sizes, color=colors, alpha=0.7)

        ax2.set_title('Basin Size Comparison', fontsize=12, fontweight='bold')
        ax2.set_ylabel('Number of Initial States', fontsize=10)
        ax2.set_xlabel('Attractors', fontsize=10)

        # Add value labels on bars
        for bar, size in zip(bars, basin_sizes):
            height = bar.get_height()
            ax2.text(bar.get_x() + bar.get_width()/2., height, f'{size}',
                     ha='center', va='bottom', fontweight='bold')

        plt.tight_layout()
        plt.show()
    return basin_data, attractor_map, n_nodes, all_states


In [156]:
# Scenario Analysis: Test all 3 scenarios from practical
# Attractor Analysis: Find all attractors for each mutated network (What percentage of states lead to cancer-like states?
# Answer the following questions in your README.md file:

#    Which mutation is most dangerous and why? Provide quantitative evidence.
#    Explain the role of feedback loops (e.g., MYC → MDM2 → p53)
#    What are the limitations of this Boolean network model? Discuss 3 specific limitations


In [157]:
# Quantitative comparison

def cancer_percentage(attractors, basin_data, node_names=node_names):
    total_states = 2 ** len(node_names)  # all initial states, including ones that never reach a steady state
    converged_states = sum(len(states) for states in basin_data.values())
    cancer_states = 0
    cancer_attractors = []

    for att_idx, attractor in enumerate(attractors):
        # Fixed points are a single state; limit cycles (from find_attractors_with_cycles) are a tuple of states
        cycle_states = attractor if isinstance(attractor[0], tuple) else (attractor,)
        state_dicts = [{node_names[j]: state[j] for j in range(len(node_names))} for state in cycle_states]

        # Cancer-like only if the cell grows (and does not die) despite DNA damage in every state of the attractor
        if all(sd['DNA_damage'] == 1 and sd['Growth'] == 1 and sd['Death'] == 0 for sd in state_dicts):
            cancer_attractors.append(att_idx)
            cancer_states += len(basin_data[att_idx])

    percentage = (cancer_states / total_states) * 100 if total_states else 0.0

    print(f"Cancer-like attractors: {len(cancer_attractors)} of {len(attractors)} "
          f"({', '.join(f'Attractor {i+1}' for i in cancer_attractors) or 'none'})")
    print(f"Initial states leading to cancer: {cancer_states} of {total_states} ({percentage}%)")
    if converged_states < total_states:
        print(f"Note: {total_states - converged_states} states never reached a steady state (limit cycle), so they are in no basin")

    return percentage


def canonical_cycle(cycle):
    """
    Make the same cycle comparable even if it is detected
    from a different starting point (rotate to the smallest version).
    """
    cycle = tuple(cycle)
    rotations = [cycle[i:] + cycle[:i] for i in range(len(cycle))]
    return min(rotations)


def find_attractors_with_cycles(network, max_steps=30):
    """
    Find all attractors (fixed points AND limit cycles) and their basins.

    Returns:
        attractors: list of attractors, each a tuple of states (length 1 = fixed point)
        basin_data: dict attractor_index -> list of initial states
    """
    node_names = sorted(network.nodes.keys())
    n_nodes = len(node_names)
    all_states = list(product([0, 1], repeat=n_nodes))
    basins = defaultdict(list)  # canonical cycle -> list of initial states

    print(f"Testing all {len(all_states)} possible initial states (fixed points and limit cycles)...")

    for initial_state in all_states:
        network.set_state(**{node_names[i]: initial_state[i] for i in range(n_nodes)})
        trajectory = network.simulate(steps=max_steps, record_history=True)
        trajectory_tuples = [tuple(int(x) for x in state) for state in trajectory]

        # The attractor starts at the first state that appears again later in the trajectory
        first_seen = {}
        for t, state in enumerate(trajectory_tuples):
            if state in first_seen:
                basins[canonical_cycle(trajectory_tuples[first_seen[state]:t])].append(initial_state)
                break
            first_seen[state] = t
        else:
            print(f"   Warning: no attractor found within {max_steps} steps for {initial_state}")

    attractors = list(basins.keys())
    basin_data = {i: basins[att] for i, att in enumerate(attractors)}
    return attractors, basin_data


def show_cycle_attractors(attractors, basin_data, node_names=node_names):
    """Print each attractor's type, basin size and states."""
    total_states = 2 ** len(node_names)
    print(f"\nFOUND {len(attractors)} ATTRACTORS:")

    for i, attractor in enumerate(attractors):
        basin_size = len(basin_data[i])
        kind = "Fixed point" if len(attractor) == 1 else f"{len(attractor)}-state limit cycle"
        print(f"\nAttractor {i+1}: {kind}, basin {basin_size} states ({basin_size / total_states * 100}%)")

        for state_number, state in enumerate(attractor):
            sd = {node_names[j]: state[j] for j in range(len(node_names))}
            print(f"   State {state_number + 1}: Growth={sd['Growth']}, Death={sd['Death']}, "
                  f"p53={sd['p53']}, p21={sd['p21']}, DNA_damage={sd['DNA_damage']}")

In [158]:
# Mutation A
net_A = make_network()  # fresh copy, so the normal network and other mutations are unaffected
net_A.add_rule('p53', lambda s: False, "p53 = BROKEN (always OFF)")  # Mutation A: p53 Knockout (Loss of tumor suppressor)

results_A = three_scenarios(net_A)
attractors_A, basin_A = find_attractors_with_cycles(net_A)
show_cycle_attractors(attractors_A, basin_A)
cancer_A = cancer_percentage(attractors_A, basin_A)


Healthy Cell
  CDK2: 1
  DNA_damage: 0
  Death: 0
  Growth: 1
  MDM2: 1
  MYC: 1
  p21: 0
  p53: 0
  -> Growth=1, Death=0, p53=0, DNA_damage=0

Stressed Cell
  CDK2: 1
  DNA_damage: 1
  Death: 0
  Growth: 1
  MDM2: 1
  MYC: 1
  p21: 0
  p53: 0
  -> Growth=1, Death=0, p53=0, DNA_damage=1

Oncogene Hijacked Cell
  CDK2: 1
  DNA_damage: 0
  Death: 0
  Growth: 1
  MDM2: 1
  MYC: 1
  p21: 0
  p53: 0
  -> Growth=1, Death=0, p53=0, DNA_damage=0
Testing all 256 possible initial states (fixed points and limit cycles)...

FOUND 2 ATTRACTORS:

Attractor 1: Fixed point, basin 128 states (50.0%)
   State 1: Growth=1, Death=0, p53=0, p21=0, DNA_damage=0

Attractor 2: Fixed point, basin 128 states (50.0%)
   State 1: Growth=1, Death=0, p53=0, p21=0, DNA_damage=1
Cancer-like attractors: 1 of 2 (Attractor 2)
Initial states leading to cancer: 128 of 256 (50.0%)


### Interpretation of Mutation A (p53 Knockout)
*Interpretation adapted from Nikolai's analysis. Google Gemini was used to help formulate and format answers.*

With p53 knocked out (always OFF):

*   **Healthy Cell**: Still reaches a growing state (Growth=1, Death=0, p53=0), similar to the original network's healthy state.
*   **Stressed Cell**: Crucially, this now leads to Growth=1, Death=0, p53=0. In the original network, a stressed cell with DNA damage would activate p53 and lead to cell death (Death=1). Here, without p53, the cell grows despite DNA damage, indicating a failure of the tumor suppression pathway.
*   **Oncogene Hijacked Cell**: Also leads to Growth=1, Death=0, p53=0, similar to the original network but now with p53 definitively unable to intervene.

**Conclusion**: The p53 knockout dramatically shifts the network's behavior, particularly under stress conditions. The cell loses its ability to undergo apoptosis (programmed cell death) in response to DNA damage, instead defaulting to an uncontrolled growth state. This confirms p53's critical role as a tumor suppressor.

In [159]:
# Mutation B
net_B = make_network()
net_B.add_rule('MYC', lambda s: True, "MYC = AMPLIFIED (always ON)")  # Mutation B: MYC Amplification (Oncogene overexpression)

results_B = three_scenarios(net_B)
attractors_B, basin_B = find_attractors_with_cycles(net_B)
show_cycle_attractors(attractors_B, basin_B)
cancer_B = cancer_percentage(attractors_B, basin_B)


Healthy Cell
  CDK2: 1
  DNA_damage: 0
  Death: 0
  Growth: 1
  MDM2: 1
  MYC: 1
  p21: 0
  p53: 0
  -> Growth=1, Death=0, p53=0, DNA_damage=0

Stressed Cell
  CDK2: 1
  DNA_damage: 1
  Death: 0
  Growth: 1
  MDM2: 1
  MYC: 1
  p21: 0
  p53: 0
  -> Growth=1, Death=0, p53=0, DNA_damage=1

Oncogene Hijacked Cell
  CDK2: 1
  DNA_damage: 0
  Death: 0
  Growth: 1
  MDM2: 1
  MYC: 1
  p21: 0
  p53: 0
  -> Growth=1, Death=0, p53=0, DNA_damage=0
Testing all 256 possible initial states (fixed points and limit cycles)...

FOUND 2 ATTRACTORS:

Attractor 1: Fixed point, basin 128 states (50.0%)
   State 1: Growth=1, Death=0, p53=0, p21=0, DNA_damage=0

Attractor 2: Fixed point, basin 128 states (50.0%)
   State 1: Growth=1, Death=0, p53=0, p21=0, DNA_damage=1
Cancer-like attractors: 1 of 2 (Attractor 2)
Initial states leading to cancer: 128 of 256 (50.0%)


### Interpretation of Mutation B (MYC Amplification)
*Interpretation adapted from Nikolai's analysis. Google Gemini was used to help formulate and format answers.*

With MYC amplified (always ON):

*   **Healthy Cell**: Leads to Growth=1, Death=0, p53=0. This is a growing state, potentially more aggressive due to constant MYC.
*   **Stressed Cell**: This is a critical observation. Despite `DNA_damage=1`, the cell ends in Growth=1, Death=0, p53=0. In the original network, this scenario led to cell death (Death=1, p53=1). MYC amplification appears to override the p53-mediated apoptosis pathway, leading to unchecked proliferation even when damaged.
*   **Oncogene Hijacked Cell**: Leads to Growth=1, Death=0, p53=0, which is expected as MYC is already amplified.

**Conclusion**: MYC amplification forces the network into a constant growth state. Even when faced with DNA damage, the cell fails to undergo apoptosis and continues to proliferate. This effectively disables the cell's natural defense mechanisms and pushes it towards an aggressive, cancerous phenotype.

In [160]:
# Mutation C
net_C = make_network()
net_C.add_rule('MDM2', lambda s: True, "MDM2 = OVEREXPRESSED (always ON)")  # Mutation C: MDM2 Overexpression (p53 pathway disruption)

results_C = three_scenarios(net_C)
attractors_C, basin_C = find_attractors_with_cycles(net_C)
show_cycle_attractors(attractors_C, basin_C)
cancer_C = cancer_percentage(attractors_C, basin_C)


Healthy Cell
  CDK2: 1
  DNA_damage: 0
  Death: 0
  Growth: 1
  MDM2: 1
  MYC: 1
  p21: 0
  p53: 0
  -> Growth=1, Death=0, p53=0, DNA_damage=0

Stressed Cell
  CDK2: 1
  DNA_damage: 1
  Death: 0
  Growth: 1
  MDM2: 1
  MYC: 1
  p21: 0
  p53: 0
  -> Growth=1, Death=0, p53=0, DNA_damage=1

Oncogene Hijacked Cell
  CDK2: 1
  DNA_damage: 0
  Death: 0
  Growth: 1
  MDM2: 1
  MYC: 1
  p21: 0
  p53: 0
  -> Growth=1, Death=0, p53=0, DNA_damage=0
Testing all 256 possible initial states (fixed points and limit cycles)...

FOUND 2 ATTRACTORS:

Attractor 1: Fixed point, basin 128 states (50.0%)
   State 1: Growth=1, Death=0, p53=0, p21=0, DNA_damage=0

Attractor 2: Fixed point, basin 128 states (50.0%)
   State 1: Growth=1, Death=0, p53=0, p21=0, DNA_damage=1
Cancer-like attractors: 1 of 2 (Attractor 2)
Initial states leading to cancer: 128 of 256 (50.0%)


### Interpretation of Mutation C (MDM2 Overexpression)

With MDM2 overexpressed (always ON):

*   **Healthy Cell**: Growing state, but the cell remains healthy.
*   **Stressed Cell**: Leads to cancer, since growth and DNA_damage are both present.
*   **Oncogene Hijacked Cell**: Cell growth but no cancer, the same as the healthy cell.

**Attractor analysis:**
Mutation C does not deviate from Mutations A and B. The cancer rate, which is checked by DNA_damage and Growth being active and Death being inactive, is the same (50%) for all of Mutations A, B and C. All simulations end up in one of two final states: one is carcinogenic and the other is not. It differs, however, from the original network, which only had a cancer rate of 3.1%. The reason is that MDM2 switches off p53, so growth never stops. In all the simulations where DNA_damage is active from the beginning, it remains active. The maximum cancer rate is therefore 50%, because DNA_damage is active in 50% of the simulations.

**Conclusion**: DNA damage at the start always leads to cancer, since p53 never has a chance to turn on due to MDM2 overexpression. Otherwise, the cell is always healthy, since DNA_damage cannot be caused by gene expression.

In [161]:
# Mutation D
net_D = make_network()
net_D.add_rule('p21', lambda s: False, "p21 = BROKEN (always OFF - Mutation D)")  # Mutation D: p21 Knockout (loss of cell cycle brake)

results_D = three_scenarios(net_D)
attractors_D, basin_D = find_attractors_with_cycles(net_D)
show_cycle_attractors(attractors_D, basin_D)
cancer_D = cancer_percentage(attractors_D, basin_D)


Healthy Cell
  CDK2: 1
  DNA_damage: 0
  Death: 0
  Growth: 1
  MDM2: 1
  MYC: 1
  p21: 0
  p53: 0
  -> Growth=1, Death=0, p53=0, DNA_damage=0

Stressed Cell
  -> No fixed steady state reached (network keeps changing, possibly a limit cycle)

Oncogene Hijacked Cell
  CDK2: 1
  DNA_damage: 0
  Death: 0
  Growth: 1
  MDM2: 1
  MYC: 1
  p21: 0
  p53: 0
  -> Growth=1, Death=0, p53=0, DNA_damage=0
Testing all 256 possible initial states (fixed points and limit cycles)...

FOUND 5 ATTRACTORS:

Attractor 1: Fixed point, basin 128 states (50.0%)
   State 1: Growth=1, Death=0, p53=0, p21=0, DNA_damage=0

Attractor 2: 3-state limit cycle, basin 56 states (21.875%)
   State 1: Growth=0, Death=0, p53=1, p21=0, DNA_damage=1
   State 2: Growth=0, Death=1, p53=1, p21=0, DNA_damage=1
   State 3: Growth=0, Death=1, p53=0, p21=0, DNA_damage=1

Attractor 3: Fixed point, basin 24 states (9.375%)
   State 1: Growth=0, Death=1, p53=1, p21=0, DNA_damage=1

Attractor 4: 3-state limit cycle, basin 40 states (

### Interpretation of Mutation D (p21 Knockout)
*Mutation D and its interpretation are Tjebbe's.*

**Scenario analysis:**
* With p21 knocked out, the Healthy Cell and Oncogene Hijacked Cell both reach a stable growth state with Growth = 1, Death = 0, p53 = 0, and p21 = 0.
* The Stressed Cell does not reach a fixed steady state. Instead, the network enters a limit cycle. This means that the values of several regulatory nodes continue to change rather than stabilizing.

**Conclusion**: In the normal network, DNA damage activates the p53 pathway and leads to a stable state. Removing p21 disrupts this response because p21 normally helps inhibit MYC and CDK2. As a result, the damaged cell no longer stabilizes into the same stable response and instead keeps oscillating.

**Attractor analysis:**
The p21 knockout did not increase the percentage of states reaching the cancer-like attractor, which remained 3.1% (8 out of 256 states). However, it strongly changed the dynamics of the network. Of the 128 states with DNA damage, only 24 settled into stable apoptosis, while 96 entered one of two 3-state limit cycles (56 + 40 states). In one of these cycles Death switches on and off, and in the other none of the outputs stay on. This means that p21 is important for stabilizing the cell's response to DNA damage. Without p21, the network often fails to settle into a stable cell-death response and instead oscillates between states.

In [162]:
# Which mutation is most dangerous?

cancer_rates = {
    'A: p53 Knockout': cancer_A,
    'B: MYC Amplification': cancer_B,
    'C: MDM2 Overexpression': cancer_C,
    'D: p21 Knockout': cancer_D,
}

print(" \n Cancer-like rates (% of all initial states):")
for name, rate in cancer_rates.items():
    print(f"  {name} {rate}%")

highest_rate = max(cancer_rates.values())
highest = [name for name, rate in cancer_rates.items() if rate == highest_rate]  # list, since several can tie
print(f" \n Highest: {', '.join(highest)} ({highest_rate}%)")

 
 Cancer-like rates (% of all initial states):
  A: p53 Knockout 50.0%
  B: MYC Amplification 50.0%
  C: MDM2 Overexpression 50.0%
  D: p21 Knockout 3.125%
 
 Highest: A: p53 Knockout, B: MYC Amplification, C: MDM2 Overexpression (50.0%)


### Which mutation is most dangerous?

A, B and C are tied at 50% cancer rate. It's hard to say, from the biology alone, which is worse. A and C seem to be effectively the same, since they both have the role of knocking out p53. One could make the argument that B and C are worse since, on top of knocking out p53, they also activate malignant genes. However, according to these models, all these mutations are equally carcinogenic.

### Role of feedback loops

In this simplified model, the side that is active first suppresses the other, with no possibility of this changing. If they all start in the off state, the first to activate suppresses the other. Here the competition is between p53 (with p21) and MYC/MDM2: p53 inhibits MYC, MYC activates MDM2, and MDM2 inhibits p53. This loop contains two inhibitions, making it a positive feedback loop, which acts as a switch with two possible outcomes: p53 on with MYC/MDM2 off, or the other way around. With DNA damage, these two outcomes are two of the network's attractors: apoptosis (120/256 states) and cancer-like growth (8/256 states). Without DNA damage, p53 can never activate, so MYC/MDM2 always wins, giving the healthy growth attractor. Mutations A–C break this loop by fixing one of its nodes, so the switch can only go one way and the cancer rate rises from 3.1% to 50%.

A negative feedback loop (an odd number of inhibitions), in contrast, causes a node to switch itself off after turning on, which can lead to oscillations where the simulation never finds a stable state, as seen in the 3-node demo network.